# 02 - On Isleme

Bu notebook, proje icindeki on isleme adimlarini adim adim dogrular:
- 0 -> NaN donusumu
- Median imputasyon
- Standardizasyon
- SMOTE ile sinif dengesi


In [ ]:
from pathlib import Path
import sys

proje_kok = Path.cwd().resolve()
while proje_kok.name != 'diyabet_risk_tahmini' and proje_kok.parent != proje_kok:
    proje_kok = proje_kok.parent

if proje_kok.name != 'diyabet_risk_tahmini':
    raise RuntimeError('Notebook proje kokunden veya altindan calistirilmali.')

if str(proje_kok) not in sys.path:
    sys.path.insert(0, str(proje_kok))

print(f'Proje koku: {proje_kok}')


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

from makine_ogrenmesi.kaynak.veri_yukleyici import veri_setini_yukle
from makine_ogrenmesi.kaynak.on_isleme import (
    sifirlari_nan_yap,
    median_imputer_olustur,
    standard_scaler_olustur,
)
from makine_ogrenmesi.kaynak.ozellik_yapilandirmasi import (
    HEDEF_KOLONU,
    OZELLIK_KOLONLARI,
    SIFIRI_EKSIK_SAYILAN_KOLONLAR,
)


In [ ]:
veri = veri_setini_yukle(proje_kok / 'makine_ogrenmesi' / 'veri' / 'ham' / 'diabetes.csv')
X = veri[OZELLIK_KOLONLARI].copy()
y = veri[HEDEF_KOLONU].copy()

print('Orijinal boyut:', X.shape)
X.head()


In [ ]:
sifir_once = (X[SIFIRI_EKSIK_SAYILAN_KOLONLAR] == 0).sum().rename('sifir_once')
X_nan = sifirlari_nan_yap(X, SIFIRI_EKSIK_SAYILAN_KOLONLAR)
nan_sonra = X_nan[SIFIRI_EKSIK_SAYILAN_KOLONLAR].isna().sum().rename('nan_sonra')

pd.concat([sifir_once, nan_sonra], axis=1)


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_nan,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print('Train boyutu:', X_train.shape)
print('Test boyutu :', X_test.shape)
print('Train sinif dagilimi:')
print(y_train.value_counts().sort_index())


In [ ]:
imputer = median_imputer_olustur()
X_train_imp = pd.DataFrame(imputer.fit_transform(X_train), columns=OZELLIK_KOLONLARI, index=X_train.index)
X_test_imp = pd.DataFrame(imputer.transform(X_test), columns=OZELLIK_KOLONLARI, index=X_test.index)

print('Train icinde kalan NaN sayisi:', int(X_train_imp.isna().sum().sum()))
print('Test icinde kalan NaN sayisi :', int(X_test_imp.isna().sum().sum()))
X_train_imp.head()


In [ ]:
scaler = standard_scaler_olustur()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train_imp), columns=OZELLIK_KOLONLARI, index=X_train_imp.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test_imp), columns=OZELLIK_KOLONLARI, index=X_test_imp.index)

istatistik = pd.DataFrame({
    'ortalama': X_train_scaled.mean(),
    'std': X_train_scaled.std(ddof=0),
})
istatistik


In [ ]:
smote = SMOTE(random_state=42)
X_train_bal, y_train_bal = smote.fit_resample(X_train_scaled, y_train)

sinif_ozet = pd.DataFrame({
    'oncesi': y_train.value_counts().sort_index(),
    'smote_sonrasi': pd.Series(y_train_bal).value_counts().sort_index(),
})
sinif_ozet.index = ['sinif_0', 'sinif_1']
sinif_ozet


## Kisa yorum

- 0 -> NaN donusumu veri kalitesini duzeltmek icin gerekli.
- Imputasyon yalnizca train verisi ile fit edilerek veri sizintisi engellenir.
- SMOTE yalnizca train tarafinda uygulanmalidir; test seti dogal dagilimda kalir.
